# 实验 22 — dbt selector 语法与 `--defer --state` 增量 CI

**目标**：掌握生产里最常用的两类“怎么只跑我关心的部分”：

1. **selector 语法**：`+model+`、`@model`、`tag:`、`config.materialized:`、`source:` 等
2. **state-aware 运行**：`--defer --state path/to/prod-manifest --select state:modified+` —— PR CI 只跑“我改过的 + 它下游”，没改的从 prod 借结果。这是 dbt 项目变大之后真正能省钱省时间的招。

In [ ]:
import subprocess, os
def dbt(*a):
    r = subprocess.run(['uv','run','dbt',*a], capture_output=True, text=True,
                       cwd='../dbt_project',
                       env={**os.environ,'DBT_PROFILES_DIR':'.'})
    return r.stdout + r.stderr

# 1) 把当前所有 model 列出来
print(dbt('ls'))

In [ ]:
# 2) 选某模型 + 下游（`+`）
print('stg_ecb_rates + downstream:')
print(dbt('ls','--select','stg_ecb_rates+'))
print()
print('fct_daily_rates + upstream:')
print(dbt('ls','--select','+fct_daily_rates'))
print()
print('整条到 fct_daily_rates 的链:')
print(dbt('ls','--select','@fct_daily_rates'))

In [ ]:
# 3) 按 tag 选 —— 先给 fct_daily_rates 加个 tag
from pathlib import Path
yml = Path('../dbt_project/models/marts/_marts.yml')
orig = yml.read_text()
yml.write_text(orig.replace(
    '- name: fct_daily_rates',
    '- name: fct_daily_rates\n    config:\n      tags: ["critical","nightly"]'
))
print(dbt('ls','--select','tag:critical'))

In [ ]:
# 4) state-aware：先做一份 'prod manifest' 副本
import shutil, os
os.makedirs('../prod_state', exist_ok=True)
# 触发一次完整 build 拿到 manifest
print(dbt('build','--exclude','snp_currencies')[-300:])
shutil.copy('../dbt_project/target/manifest.json', '../prod_state/manifest.json')
print('snapshot prod manifest:', os.path.getsize('../prod_state/manifest.json'), 'bytes')

In [ ]:
# 5) 模拟 PR 改动：改 stg_ecb_rates.sql 加注释（任何改动都会让它进 state:modified）
stg = Path('../dbt_project/models/staging/stg_ecb_rates.sql')
stgorig = stg.read_text()
stg.write_text('-- PR-comment-' + 'X' + '\n' + stgorig)
print(dbt('ls','--select','state:modified','--state','../prod_state'))

In [ ]:
# 6) 真正的 CI 模式：state:modified+ 表示“改了的 + 下游全部”
print(dbt('ls','--select','state:modified+','--state','../prod_state'))
print()
print('这就是 slim CI：PR 上只跑这几个，没改的从 prod 拿结果（defer）')

In [ ]:
# 还原文件
stg.write_text(stgorig)
yml.write_text(orig)
import shutil
shutil.rmtree('../prod_state', ignore_errors=True)

## 生产环境踩坑提示

- **slim CI 的关键**：CI runner 里 (1) 下载最新 prod manifest（一般 dbt artifacts 上传到 S3） (2) 用它当 `--state` (3) 跑 `dbt build --defer --state ./prod_state --select state:modified+`。能 30 分钟的 CI 缩到 3 分钟。
- **`+` 数字**：`model+2` = 下游 2 层，`+2model` = 上游 2 层。不写就是无穷。
- **`state:modified` 检测的是什么**：模型 SQL、配置、依赖、文档（看你怎么配 modifiers）。光改 description 也算 modified（如不想算，加 `--exclude-resource-type analysis` 或调 `state:modified.body` 之类的细粒度修饰）。
- **`defer` 的本质**：当一个 ref 指向没 build 的 model 时，从 `--state` 提供的 manifest 找它在 prod 的真实表名，rewrite 成 prod 的引用。所以 prod 必须真的有那些表。
- **selector 文件**：复杂选择写在 `selectors.yml` 里，命名后 `--selector my_critical_models` 调用，跟 PR 模板配合很好。

## 思考题

- 想跑“所有 tag=critical 的模型 + 它们的全部上游”：怎么写 selector？（`+tag:critical`）
- `state:modified.persisted_descriptions` 是什么意思？哪些 modifier 在 CI 上最常用？
- 你公司 CI 现在是“PR 跑全量” 还是 “slim CI”？如果是前者，估一估每月浪费多少 warehouse 钱？